# 01. Data audit

Loads the exact cached snapshots, checks coverage, dates, missing values,
duplicates, price validity, and external-series coverage. Uses `src/` only.
no model code here.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from coal_forecasting.config import load_config
from coal_forecasting.data import load_snapshots

config = load_config(Path('configs/baseline.toml'))
snaps = load_snapshots(config)  # reuses cache; fails on sidecar mismatch
for symbol, snap in snaps.items():
    f = snap.frame
    print(symbol, f['Date'].min().date(), '->', f['Date'].max().date(),
          f'rows={len(f)}', 'retrieved:', snap.retrieved_at_utc)

In [ ]:
for symbol, snap in snaps.items():
    f = snap.frame
    assert not f['Date'].duplicated().any(), symbol
    assert f[['Close', 'Adj Close']].notna().all().all(), symbol
    assert np.isfinite(f[['Close', 'Adj Close']].to_numpy(float)).all(), symbol
    assert (f[['Close', 'Adj Close']] > 0).all().all(), symbol
    print(symbol, 'ok: unique dates, no missing/non-finite/non-positive prices')

In [ ]:
# Trading-calendar overlap: which equity dates lack a same-day FX observation?
eq = snaps['ADRO.JK'].frame[['Date']]
fx = set(snaps['IDR=X'].frame['Date'])
missing = eq[~eq['Date'].isin(fx)]
print('ADRO dates without same-day FX row:', len(missing))
print('Handled by backward as-of alignment + one-observation lag (see docs/preprocessing.md).')